In [1]:
import os
from PIL import Image

# 嘗試導入 pdf2image（若未安裝則針對 PDF 轉圖片提示錯誤）
try:
    from pdf2image import convert_from_path
    HAS_PDF2IMAGE = True
except ImportError:
    HAS_PDF2IMAGE = False

def convert_image():
    # 1. 輸入設定
    path = input("請輸入圖片或 PDF 路徑/資料夾路徑: ").strip('"').strip("'")
    if not os.path.exists(path):
        print("❌ 錯誤：找不到該路徑！")
        return

    # 2. 選擇目標格式 (新增 pdf)
    target_format = input("想要的目標格式 (png, jpg, webp, ico, bmp, pdf): ").lower().strip('.')

    if target_format not in ["png", "jpg", "webp", "ico", "bmp", "pdf"]:
        print("❌ 不支援的目標格式！")
        return

    # 支援的原始副檔名 (新增 .pdf)
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.gif', '.jfif', '.pdf')

    # 判斷是單一檔案還是資料夾
    files_to_process = []
    if os.path.isfile(path):
        if path.lower().endswith(valid_extensions):
            files_to_process.append(path)
    elif os.path.isdir(path):
        for f in os.listdir(path):
            if f.lower().endswith(valid_extensions):
                files_to_process.append(os.path.join(path, f))

    if not files_to_process:
        print("⚠️ 找不到可處理的檔案。")
        return

    print(f"\n🚀 開始處理 {len(files_to_process)} 個檔案...")
    success_count = 0

    for input_path in files_to_process:
        try:
            file_base = os.path.splitext(input_path)[0]
            output_path = f"{file_base}.{target_format}"
            is_input_pdf = input_path.lower().endswith('.pdf')

            # 防止原地覆蓋
            if input_path.lower() == output_path.lower():
                print(f"⏩ 跳過相同格式: {os.path.basename(input_path)}")
                continue

            # ------------------------------------------------------------------
            # 情境 A：來源是 PDF 檔 ➔ 轉為圖片格式
            # ------------------------------------------------------------------
            if is_input_pdf:
                if not HAS_PDF2IMAGE:
                    print(f"❌ 無法處理 {os.path.basename(input_path)}：未安裝 pdf2image 套件。")
                    continue

                # 將 PDF 每頁轉為 PIL Image
                pages = convert_from_path(input_path)
                for idx, page in enumerate(pages):
                    # 若只有一頁則維持原檔名，多頁則加上頁碼標籤
                    page_output = output_path if len(pages) == 1 else f"{file_base}_page_{idx+1}.{target_format}"
                    
                    if target_format in ["jpg", "jpeg"]:
                        if page.mode in ("RGBA", "P"):
                            page = page.convert("RGB")

                    page.save(page_output, format=target_format.upper())
                
                print(f"✅ 已轉換 PDF ({len(pages)} 頁): {os.path.basename(input_path)} -> {target_format}")
                success_count += 1
                continue

            # ------------------------------------------------------------------
            # 情境 B：來源是圖片檔
            # ------------------------------------------------------------------
            with Image.open(input_path) as img:

                # 目標格式為 PDF 時
                if target_format == "pdf":
                    if img.mode in ("RGBA", "P"):
                        img = img.convert("RGB")
                    img.save(output_path, format="PDF")

                # 目標格式為 JPG / JPEG 時
                elif target_format in ["jpg", "jpeg"]:
                    if img.mode in ("RGBA", "P"):
                        img = img.convert("RGB")
                    img.save(output_path, format="JPEG")

                # 目標格式為 ICO 時
                elif target_format == "ico":
                    if img.width > 256 or img.height > 256:
                        img.thumbnail((256, 256), Image.Resampling.LANCZOS)
                    icon_sizes = [(16, 16), (32, 32), (48, 48), (64, 64), (128, 128), (256, 256)]
                    img.save(output_path, format="ICO", sizes=icon_sizes)

                # 其他一般圖片格式 (PNG, WEBP, BMP 等)
                else:
                    img.save(output_path, format=target_format.upper())

            print(f"✅ 已轉換: {os.path.basename(input_path)} -> {target_format}")
            success_count += 1

        except Exception as e:
            print(f"❌ 處理 {os.path.basename(input_path)} 時發生錯誤: {e}")

    print(f"\n✨ 任務結束！成功轉換 {success_count} 個檔案。")

if __name__ == "__main__":
    convert_image()


🚀 開始處理 1 個檔案...
✅ 已轉換: t1.png -> pdf

✨ 任務結束！成功轉換 1 個檔案。
